# North Atlantic surface heat and freshwater fluxes

Compares the surface heat and freshwater fluxes of ACCESS-OM3 and ACCESS-OM2 (`025deg_jra55_iaf_omip2_cycle1`) in the North Atlantic with each other and with the ERA5 reanalysis: time-mean maps, annual-mean time series in the Labrador and GIN Seas, and a breakdown of the models' net freshwater flux there into P − E, rivers, icebergs and sea ice.

Context: low-AMOC bias, see [#7](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/7), [#195](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/195) and [#196](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/196). Deep convection in the Labrador Sea stops early in the ACCESS-OM3 run (see `SSS_MLD_Labrador_GIN_timeseries.ipynb`); this notebook asks how the surface forcing differs there.

All fluxes are positive into the ocean.

| | ACCESS-OM3 | ACCESS-OM2 | ERA5 |
|---|---|---|---|
| Net heat flux | `hfds` | `net_sfc_heating` + `frazil_3d_int_z` | `msnswrf` + `msnlwrf` + `mslhf` + `msshf` |
| Precipitation − evaporation (P − E) | `precip` + `evs` (daily) | `lprec` + `fprec` + `evap` | `mtpr` + `mer` |
| Net freshwater flux (P − E + runoff + sea-ice melt/freeze) | `wfo` | `pme_river` | — |
| SSS restoring (a salt flux in both models) | `salt_flux_added` | `sfc_salt_flux_restore` | — |

**Caveats on ERA5 as the reference.** ERA5 fluxes come from a reanalysis with observed SST and sea ice, not from flux measurements, and use the ECMWF bulk formulae; the models are forced by JRA55-do with the NCAR bulk formulae, so part of any model–ERA5 difference is the forcing and bulk formulae rather than the ocean model. Over sea ice ERA5 gives the flux at the ice or snow surface, whereas the models give the flux into the ocean, so the comparison is restricted to points that are mostly ice-free in ERA5 (see below). ERA5 has no runoff or sea-ice meltwater, so the total freshwater flux and SSS restoring are compared between the models only.

In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test5.1-8f50b769/datastore.json"

# papermill settings. *No need to modify these if running interactively.*
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory
nbname = None                          # notebook name

In [ ]:
if not papermill:
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)

In [ ]:
import glob
import numpy as np
import intake
import xarray as xr
import xesmf
import cmocean
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from distributed import Client
from exptdata_access import get_exptname_from_path

client = Client(threads_per_worker=1)
om3exptname = get_exptname_from_path(esm_file)
om2exptname = "025deg_jra55_iaf_omip2_cycle1"

# Period for the maps: the last 25 years both runs cover (OM2 IAF cycle 1 ends in 2018)
period = slice("1994", "2018")

## Unit conversions

Freshwater fluxes are shown in m yr⁻¹ of liquid water. The restoring salt flux $F_S$ (kg salt m⁻² s⁻¹) is converted to the freshwater flux that would change salinity at the same rate, $-F_S / (S_0/1000)$ with $S_0 = 35$ g kg⁻¹; it is negative where restoring adds salt (acting like evaporation).

In [ ]:
seconds_per_year = 365.25 * 86400
rho_fw = 1000.0  # kg m-3
S0 = 35.0        # g kg-1, reference salinity for the restoring conversion

def to_m_per_year(water_flux):  # kg m-2 s-1 -> m yr-1
    return water_flux / rho_fw * seconds_per_year

def salt_to_freshwater(salt_flux):  # kg salt m-2 s-1 -> equivalent freshwater flux, kg m-2 s-1
    return -salt_flux / (S0 / 1000)

quantities = {  # quantity: (title, units)
    "heat": ("Net surface heat flux", "W m$^{-2}$"),
    "pme": ("Precipitation − evaporation", "m yr$^{-1}$"),
    "freshwater": ("Net surface freshwater flux", "m yr$^{-1}$"),
    "restoring": ("SSS restoring as freshwater flux", "m yr$^{-1}$"),
}

## Restrict everything to the North Atlantic

All three datasets are cut to 100°W–30°E, 0–82°N straight after loading, so only that part of each file is read. The helpers select the smallest index box containing a lon/lat box on any grid (curvilinear or not).

In [ ]:
def box(lon, lat, lon_min, lon_max, lat_min, lat_max):
    lon = (lon + 180) % 360 - 180  # wrap to [-180, 180) so boxes work on any grid convention
    return (lon >= lon_min) & (lon <= lon_max) & (lat >= lat_min) & (lat <= lat_max)

def bounding_index_box(mask):
    """isel() dict selecting the smallest index box that contains all True points of a 2D mask."""
    inside = mask.values
    rows, cols = inside.any(axis=1).nonzero()[0], inside.any(axis=0).nonzero()[0]
    return {mask.dims[0]: slice(rows[0], rows[-1] + 1), mask.dims[1]: slice(cols[0], cols[-1] + 1)}

north_atlantic = (-100, 30, 0, 82)

## Load ACCESS-OM3 monthly fluxes (P and E are daily, averaged to monthly)

In [ ]:
om3cat = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=["variable", "variable_long_name", "variable_standard_name",
                            "variable_cell_methods", "variable_units"],
)

om3area = xr.open_dataset(om3cat.search(variable="areacello", frequency="fx").df.path.iloc[0])["areacello"]
# geolon/geolat from the geometry file: the static file's copies are NaN over land-only processors
om3geom = xr.open_dataset(om3cat.search(variable="geolon", filename="access-om3.mom6.geometry.nc").df.path.iloc[0])
om3geom = om3geom.rename({"lonh": "xh", "lath": "yh"}).assign_coords(xh=om3area.xh, yh=om3area.yh)
om3sub = bounding_index_box(box(om3geom["geolon"], om3geom["geolat"], *north_atlantic))
om3area = om3area.isel(om3sub)
om3lon, om3lat = om3geom["geolon"].isel(om3sub), om3geom["geolat"].isel(om3sub)

def load_om3(variable, frequency="1mon"):
    da = om3cat.search(variable=variable, frequency=frequency).to_dask(
        xarray_open_kwargs=dict(chunks={"time": 12}, decode_timedelta=True),
        xarray_combine_by_coords_kwargs=dict(compat="override", data_vars="minimal", coords="minimal"),
    )[variable].isel(om3sub)
    return da.resample(time="MS").mean() if frequency == "1day" else da

om3 = {
    "heat": load_om3("hfds"),
    "pme": to_m_per_year(load_om3("precip", "1day") + load_om3("evs", "1day")),
    "freshwater": to_m_per_year(load_om3("wfo")),
    "restoring": to_m_per_year(salt_to_freshwater(load_om3("salt_flux_added"))),
}

## Load ACCESS-OM2 monthly fluxes

In [ ]:
om2cat = intake.cat.access_nri[om2exptname]

# reference grid file: the run's own ocean_grid.nc has NaN geolon/geolat over land-only processors
om2grid = xr.open_dataset("/g/data/ik11/grids/ocean_grid_025.nc")
om2sub = bounding_index_box(box(om2grid["geolon_t"], om2grid["geolat_t"], *north_atlantic))
om2area = om2grid["area_t"].isel(om2sub)
om2lon, om2lat = om2grid["geolon_t"].isel(om2sub), om2grid["geolat_t"].isel(om2sub)

def load_om2(variable):
    return om2cat.search(variable=variable, frequency="1mon").to_dask(
        xarray_open_kwargs=dict(chunks={"time": 12}, decode_timedelta=True),
        xarray_combine_by_coords_kwargs=dict(compat="override", data_vars="minimal", coords="minimal"),
    )[variable].isel(om2sub)

om2 = {
    "heat": load_om2("net_sfc_heating") + load_om2("frazil_3d_int_z"),
    # despite its long_name, lprec excludes sea-ice melt/formation water (that is in `melt`):
    # pme_river = lprec + fprec + evap + runoff + melt
    "pme": to_m_per_year(load_om2("lprec") + load_om2("fprec") + load_om2("evap")),
    "freshwater": to_m_per_year(load_om2("pme_river")),
    "restoring": to_m_per_year(salt_to_freshwater(load_om2("sfc_salt_flux_restore"))),
}

## Load ERA5 monthly means

ERA5 single-level monthly means from `/g/data/rt52`, on a regular 0.25° grid (longitudes −180 to 180, latitudes decreasing).

In [ ]:
def load_era5(variable, years=slice(1959, 2023)):
    files = sorted(f for y in range(years.start, years.stop + 1)
                   for f in glob.glob(f"/g/data/rt52/era5/single-levels/monthly-averaged/{variable}/{y}/*.nc"))
    ds = xr.open_mfdataset(files, combine="by_coords", parallel=True,
                           data_vars="minimal", coords="minimal", compat="override")
    da = ds[list(ds.data_vars)[0]]
    return da.sel(longitude=slice(north_atlantic[0], north_atlantic[1]),
                  latitude=slice(north_atlantic[3], north_atlantic[2]))

era5 = {
    "heat": load_era5("msnswrf") + load_era5("msnlwrf") + load_era5("mslhf") + load_era5("msshf"),
    "pme": to_m_per_year(load_era5("mtpr") + load_era5("mer")),
}

era5lat, era5lon = xr.broadcast(era5["heat"].latitude, era5["heat"].longitude)
era5area = np.cos(np.deg2rad(era5lat))  # relative cell area of a regular lat-lon grid

## Ocean, mostly-ice-free mask

ERA5 points count as ocean where the land–sea mask is below 0.5, and as mostly ice-free where the 1994–2018 mean sea-ice concentration is below 15%. The mask is carried to each model grid by nearest-neighbour regridding, and applied to all three datasets in the maps and the regional means, so they are always averaged over the same area.

In [ ]:
era5ocean = (load_era5("lsm", slice(1994, 1994)).isel(time=0) < 0.5).load()
era5ice = load_era5("ci", slice(1994, 2018)).mean("time").load()
era5mask = era5ocean & (era5ice < 0.15)

era5grid = xr.Dataset(coords={"lon": era5lon, "lat": era5lat})
om3grid = xr.Dataset(coords={"lon": om3lon, "lat": om3lat})
om2grid_ds = xr.Dataset(coords={"lon": om2lon, "lat": om2lat})

def nearest_mask(target_grid):
    regrid = xesmf.Regridder(era5grid, target_grid, "nearest_s2d")
    return regrid(era5mask.astype(float)) > 0.5

om3mask = nearest_mask(om3grid)
om2mask = nearest_mask(om2grid_ds)

## Time-mean maps

Average over `period`, then regrid ACCESS-OM2 and ERA5 onto the ACCESS-OM3 grid (bilinear, using each grid's true 2D longitude/latitude) so differences can be taken point by point. Grey shading in the ERA5 comparisons marks points outside the ocean, mostly-ice-free mask.

In [ ]:
om3mean = {k: da.sel(time=period).mean("time").load() for k, da in om3.items()}
om2mean = {k: da.sel(time=period).mean("time").load() for k, da in om2.items()}
era5mean = {k: da.sel(time=period).mean("time").load() for k, da in era5.items()}

om2_to_om3 = xesmf.Regridder(om2grid_ds, om3grid, "bilinear", unmapped_to_nan=True)
era5_to_om3 = xesmf.Regridder(era5grid, om3grid, "bilinear", unmapped_to_nan=True)
om2_on_om3 = {k: om2_to_om3(da) for k, da in om2mean.items()}
era5_on_om3 = {k: era5_to_om3(da).where(om3mask & om3mean[k].notnull()) for k, da in era5mean.items()}

In [ ]:
proj = ccrs.LambertConformal(central_longitude=-35, central_latitude=45, standard_parallels=(30, 60))

def plot_maps(quantity, panels, vlim, dlim):
    """Three maps on the OM3 grid: panels is a list of (field, title, is_difference)."""
    title, units = quantities[quantity]
    fig, axs = plt.subplots(1, 3, figsize=(16, 5), subplot_kw=dict(projection=proj), layout="constrained")
    for ax, (da, name, is_difference) in zip(axs, panels):
        lim = dlim if is_difference else vlim
        ax.set_facecolor("lightgray")
        p = ax.pcolormesh(om3lon, om3lat, da, cmap=cmocean.cm.balance, vmin=-lim, vmax=lim,
                          transform=ccrs.PlateCarree())
        ax.set_extent([-90, 25, 5, 80], crs=ccrs.PlateCarree())
        ax.coastlines(resolution="110m", linewidth=0.5)
        ax.set_title(name)
        fig.colorbar(p, ax=ax, orientation="horizontal", shrink=0.8, extend="both", label=f"{title} ({units})")
    fig.suptitle(f"{title}, {period.start}–{period.stop} mean (positive into ocean)")
    return fig

def plot_vs_era5(quantity, vlim, dlim):
    obs = era5_on_om3[quantity]
    return plot_maps(quantity, [
        (obs, "ERA5", False),
        (om3mean[quantity].where(obs.notnull()) - obs, "ACCESS-OM3 − ERA5", True),
        (om2_on_om3[quantity].where(obs.notnull()) - obs, "ACCESS-OM2 − ERA5", True),
    ], vlim, dlim)

def plot_om3_vs_om2(quantity, vlim, dlim):
    return plot_maps(quantity, [
        (om3mean[quantity], "ACCESS-OM3", False),
        (om2_on_om3[quantity], "ACCESS-OM2", False),
        (om3mean[quantity] - om2_on_om3[quantity], "ACCESS-OM3 − ACCESS-OM2", True),
    ], vlim, dlim)

issue_link = "GitHub issue: [#196](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/196)"

### Net surface heat flux compared with ERA5

In [ ]:
fig = plot_vs_era5("heat", vlim=200, dlim=60)
mkmd.savefig(fig, "North Atlantic surface heat flux vs ERA5",
             f"{period.start}-{period.stop} mean net surface heat flux into the ocean in ERA5, and the ACCESS-OM3 {om3exptname} "
             f"and ACCESS-OM2 {om2exptname} differences from it, on the OM3 grid. Grey: land, or ERA5 mean sea-ice concentration above 15%. "
             + issue_link, dpi=150)

### Precipitation − evaporation compared with ERA5

In [ ]:
fig = plot_vs_era5("pme", vlim=2, dlim=0.5)
mkmd.savefig(fig, "North Atlantic P-E vs ERA5",
             f"{period.start}-{period.stop} mean precipitation minus evaporation in ERA5, and the ACCESS-OM3 {om3exptname} "
             f"and ACCESS-OM2 {om2exptname} differences from it, on the OM3 grid. Grey: land, or ERA5 mean sea-ice concentration above 15%. "
             + issue_link, dpi=150)

### Net surface freshwater flux (models only; includes runoff and sea-ice melt/freeze)

In [ ]:
fig = plot_om3_vs_om2("freshwater", vlim=2, dlim=1)
mkmd.savefig(fig, "North Atlantic surface freshwater flux",
             f"{period.start}-{period.stop} mean net surface freshwater flux into the ocean (P-E+runoff+sea-ice melt/freeze): "
             f"ACCESS-OM3 {om3exptname} (wfo), ACCESS-OM2 {om2exptname} (pme_river), and their difference. " + issue_link, dpi=150)

### SSS restoring, as an equivalent freshwater flux (models only)

In [ ]:
fig = plot_om3_vs_om2("restoring", vlim=0.5, dlim=0.5)
mkmd.savefig(fig, "North Atlantic SSS restoring flux",
             f"{period.start}-{period.stop} mean SSS restoring salt flux expressed as an equivalent freshwater flux (negative = adds salt): "
             f"ACCESS-OM3 {om3exptname} (salt_flux_added), ACCESS-OM2 {om2exptname} (sfc_salt_flux_restore), and their difference. "
             + issue_link, dpi=150)

## Time series in the convection regions

Area-weighted means over the same regions as `SSS_MLD_Labrador_GIN_timeseries.ipynb`, using only points inside the ocean, mostly-ice-free mask, then annual means (each month weighted by its length).

- **Labrador Sea**: 53–65°N, 65–43°W
- **GIN Seas** (Greenland–Iceland–Norwegian): 65–80°N, 20°W–15°E

In [ ]:
regions = {  # lon_min, lon_max, lat_min, lat_max (degrees east / north)
    "Labrador Sea": (-65, -43, 53, 65),
    "GIN Seas": (-20, 15, 65, 80),
}

def regional_annual_mean(da, area, lon, lat, region_box):
    in_region = box(lon, lat, *region_box)
    subset = bounding_index_box(in_region)  # only read the region's bounding box
    da, area, in_region = da.isel(subset), area.isel(subset), in_region.isel(subset)
    weights = area.where(in_region & da.isel(time=0).notnull(), 0.0)
    monthly = da.weighted(weights).mean(list(area.dims))
    days = monthly.time.dt.days_in_month
    return ((monthly * days).groupby("time.year").sum() / days.groupby("time.year").sum()).load()

datasets = {  # name: (fluxes, masked cell area, lon, lat)
    "ACCESS-OM3": (om3, om3area.where(om3mask, 0.0), om3lon, om3lat),
    "ACCESS-OM2": (om2, om2area.where(om2mask, 0.0), om2lon, om2lat),
    "ERA5": (era5, era5area.where(era5mask, 0.0), era5lon, era5lat),
}

timeseries = {
    quantity: {
        region: {
            name: regional_annual_mean(fluxes[quantity], area, lon, lat, region_box)
            for name, (fluxes, area, lon, lat) in datasets.items() if quantity in fluxes
        }
        for region, region_box in regions.items()
    }
    for quantity in quantities
}

In [ ]:
styles = {"ACCESS-OM3": dict(color="C0"), "ACCESS-OM2": dict(color="C1"), "ERA5": dict(color="k", ls="--")}

fig, axs = plt.subplots(len(quantities), 2, figsize=(12, 12), sharex=True, layout="constrained")
for row, (quantity, (title, units)) in enumerate(quantities.items()):
    for col, region in enumerate(regions):
        ax = axs[row, col]
        for name, ts in timeseries[quantity][region].items():
            ts.plot(ax=ax, label=name, **styles[name])
        ax.axhline(0, color="k", lw=0.5)
        ax.set_title(f"{region}: {title}", fontsize=10)
        ax.set_ylabel(units)
        ax.set_xlabel("Year" if row == len(quantities) - 1 else "")
axs[0, 0].legend()
mkmd.savefig(fig, "Labrador and GIN Seas surface fluxes",
             "Annual-mean, area-averaged net surface heat flux, precipitation minus evaporation, net surface freshwater flux and "
             "SSS restoring (as equivalent freshwater flux) in the Labrador Sea (53-65°N, 65-43°W) and GIN Seas (65-80°N, 20°W-15°E), "
             f"positive into the ocean, over points that are ocean and mostly ice-free in ERA5: ACCESS-OM3 {om3exptname}, "
             f"ACCESS-OM2 {om2exptname} and ERA5 (dashed; heat flux and P-E only). " + issue_link, dpi=150)

## Regional means over the common period

Mean of the annual values over `period`, for a quick comparison of the three datasets.

In [ ]:
import pandas as pd

years = range(int(period.start), int(period.stop) + 1)
table = pd.DataFrame({
    (region, name): {quantity: float(timeseries[quantity][region][name].sel(year=years).mean())
                     for quantity in quantities if name in timeseries[quantity][region]}
    for region in regions for name in datasets
}).round(2)
table.index = [f"{quantities[q][0]} ({quantities[q][1]})".replace("$^{-2}$", "⁻²").replace("$^{-1}$", "⁻¹") for q in table.index]
mkmd.table(f"Labrador and GIN Seas mean surface fluxes, {period.start}-{period.stop}", table.to_markdown().split("\n"))
table

## Components of the net freshwater flux

Which terms make up the net freshwater flux, and which of them differ between the models? Each model's net flux splits exactly into the terms below (checked to round-off on a sample month):

- ACCESS-OM3: `wfo` = `precip` + `evs` (P − E) + `friver` (rivers) + `ficeberg` (icebergs) + `fsitherm` (sea-ice melt − formation)
- ACCESS-OM2: `pme_river` = `lprec` + `fprec` + `evap` (P − E) + `runoff` + `melt` (sea-ice melt − formation). ACCESS-OM2 has no separate iceberg term, so all its land freshwater is in `runoff`.

Unlike the sections above, these are averaged over all ocean points in each region, including those under sea ice and the coastal points where runoff enters, so the sea-ice and runoff terms are complete. The net flux here therefore differs from the mostly-ice-free values in the table above.

In [ ]:
components = {  # component: {model: flux in m yr-1}
    "P − E": {"ACCESS-OM3": om3["pme"], "ACCESS-OM2": om2["pme"]},
    "Rivers": {"ACCESS-OM3": to_m_per_year(load_om3("friver")), "ACCESS-OM2": to_m_per_year(load_om2("runoff"))},
    "Icebergs": {"ACCESS-OM3": to_m_per_year(load_om3("ficeberg"))},
    "Sea-ice melt − formation": {"ACCESS-OM3": to_m_per_year(load_om3("fsitherm", "1day")),
                                 "ACCESS-OM2": to_m_per_year(load_om2("melt"))},
    "Net": {"ACCESS-OM3": om3["freshwater"], "ACCESS-OM2": om2["freshwater"]},
}
grids = {"ACCESS-OM3": (om3area, om3lon, om3lat), "ACCESS-OM2": (om2area, om2lon, om2lat)}  # unmasked areas

component_ts = {
    component: {
        region: {name: regional_annual_mean(da, *grids[name], region_box) for name, da in by_model.items()}
        for region, region_box in regions.items()
    }
    for component, by_model in components.items()
}

In [ ]:
component_colors = {"P − E": "C0", "Rivers": "C2", "Icebergs": "C4", "Sea-ice melt − formation": "C1", "Net": "k"}
model_styles = {"ACCESS-OM3": "-", "ACCESS-OM2": "--"}

fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharey=True, layout="constrained")
for ax, region in zip(axs, regions):
    for component, color in component_colors.items():
        for name, ts in component_ts[component][region].items():
            ts.plot(ax=ax, color=color, ls=model_styles[name], label=f"{component} ({name})")
    ax.axhline(0, color="k", lw=0.5)
    ax.set_title(region)
    ax.set_ylabel("m yr$^{-1}$")
    ax.set_xlabel("Year")
fig.legend(*axs[0].get_legend_handles_labels(), loc="outside lower center", ncol=5, fontsize=8)
mkmd.savefig(fig, "Labrador and GIN Seas freshwater flux components",
             "Annual-mean, area-averaged components of the net surface freshwater flux (positive into the ocean) over all ocean "
             "points in the Labrador Sea (53-65°N, 65-43°W) and GIN Seas (65-80°N, 20°W-15°E): P-E, rivers, icebergs, "
             "sea-ice melt minus formation, and the net flux. "
             f"Solid: ACCESS-OM3 {om3exptname}; dashed: ACCESS-OM2 {om2exptname}, whose rivers term includes all land "
             "freshwater and which has no separate iceberg term. " + issue_link, dpi=150)

In [ ]:
component_table = pd.DataFrame({
    (region, name): {component: float(component_ts[component][region][name].sel(year=years).mean())
                     for component in components if name in component_ts[component][region]}
    for region in regions for name in grids
})
for region in regions:
    component_table[(region, "OM3 − OM2")] = (component_table[(region, "ACCESS-OM3")]
                                              - component_table[(region, "ACCESS-OM2")])
component_table = component_table[[(r, c) for r in regions for c in [*grids, "OM3 − OM2"]]].round(2)
mkmd.table(f"Labrador and GIN Seas freshwater flux components (m yr⁻¹), {period.start}-{period.stop}",
           component_table.to_markdown().split("\n"))
component_table

In [ ]:
client.close()